<a href="https://colab.research.google.com/github/sebauhlf/Proyecto-ML-IPLN/blob/main/Clasificacion_de_titulares.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto de clasificacion de clickbaits

---


## Carga de datos

Los datos utilizados en este proyecto fueron usados en una [competencia](https://www.fing.edu.uy/inco/grupos/pln/ta1c/) internacional que tuvo lugar entre marzo y junio de 2025. El objetivo de la competencia, llamada TA1C (Te Ahorré Un Click), era impulsar el desarrollo de sistemas de PLN que pudieran detectar y arruinar clickbait. En este laboratorio nos vamos a centrar únicamente en detectar cuándo un titular de una noticia es clickbait o no, lo que se corresponde con un problema de clasificación binaria (de dos clases).

El origen de estos datos está en la [tesis de maestría](https://hdl.handle.net/20.500.12008/48614) de Gabriel Mordecki, que también fueron detallados en su respectiva publicación "[Te Ahorré Un Click: A Revised Definition of Clickbait and Detection in Spanish News](https://link.springer.com/chapter/10.1007/978-3-031-80366-6_32)".

En esta primera sección del *notebook* se cargan los datos en cuatro listas:
1. `train_headlines` va a contener todos los titulares de noticias del conjunto de *train*.
2. `train_clickbait` va a contener las anotaciones del conjunto de *train*, indicando si el correspondiente titular es clickbait o no (`train_clickbait[i]` es la anotación del titular `train_headlines[i]`).
3. `dev_headlines` que va a contener lo análogo a `train_headlines` pero esta vez siendo del conjunto de desarrollo (*dev*).
4. `dev_clickbait` que va a contener lo análogo a `train_clickbait` pero esta vez siendo del conjunto de desarrollo (*dev*).


In [2]:
%%capture
import csv

!wget https://raw.githubusercontent.com/pln-fing-udelar/pln-inco-resources/refs/heads/master/clickbait/iberlef2025/detection/TA1C_dataset_detection_train.csv
!wget https://raw.githubusercontent.com/pln-fing-udelar/pln-inco-resources/refs/heads/master/clickbait/iberlef2025/detection/TA1C_dataset_detection_dev_gold.csv

with open("TA1C_dataset_detection_train.csv","r") as f:
    train = [x for x in csv.reader(f)]

with open("TA1C_dataset_detection_dev_gold.csv","r") as f:
    dev = [x for x in csv.reader(f)]

train_headlines = [x[4] for x in train[1:]]
train_clickbait = [x[5] for x in train[1:]]

dev_headlines = [x[4] for x in dev[1:]]
dev_clickbait = [x[6] for x in dev[1:]]

Prueba para verificar que los datos hayan cargados correctamente:

In [ ]:
index_train = 24
print(f'El titular en la posición {index_train} en TRAIN es "{train_headlines[index_train]}"\n¿Clickbait? --> {train_clickbait[index_train]}\n\n')

index_dev = 39
print(f'El titular en la posición {index_dev} en DEV es "{dev_headlines[index_dev]}"\n¿Clickbait? --> {dev_clickbait[index_dev]}')

El titular en la posición 24 en TRAIN es "Sin TACC: todo lo que necesitas saber sobre las comidas para celíacos"
¿Clickbait? --> Clickbait


El titular en la posición 39 en DEV es "El consumo de sidra creció y ahora se consume durante todo el año"
¿Clickbait? --> No


## 1️⃣Clasificación con modelos estadísticos sin usar embeddings como entrada


---

En esta parte inicial del laboratorio, se utilizan modelos estadísticos como SVC y Regresión Logística para clasificar los titulares en el conjunto Dev en las clases "Clickbait" y "No Clickbait".

TF-IDF()

LogisticRegression(max_iter=1000, class_weight='balanced')

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

TF_IDF_vectorizer = TfidfVectorizer()


LR = LogisticRegression(max_iter=1000, class_weight='balanced')


training_features = TF_IDF_vectorizer.fit_transform(train_headlines) # Se vectorizan los textos de train
LR.fit(training_features, train_clickbait) # Se entrena el clasificador usando los textos vectorizados

dev_features = TF_IDF_vectorizer.transform(dev_headlines) # Se vectorizan los textos de dev
prediction = LR.predict(dev_features) # Se realiza la prediccion

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, prediction, average='macro')*100, 2))}\n")  # Se calcula el F1
print(classification_report(dev_clickbait, prediction))

F1-Score macro: 77.53

              precision    recall  f1-score   support

   Clickbait       0.69      0.67      0.68       203
          No       0.87      0.88      0.87       497

    accuracy                           0.82       700
   macro avg       0.78      0.77      0.78       700
weighted avg       0.82      0.82      0.82       700



TF-IDF()

SVC(C=2.0, kernel='linear')

In [ ]:
svc = SVC(C=2.0, kernel='linear') #Support Vector Machines

training_features = TF_IDF_vectorizer.fit_transform(train_headlines) # Se vectorizan los textos de train
svc.fit(training_features, train_clickbait) # Se entrena el clasificador usando los textos vectorizados

dev_features = TF_IDF_vectorizer.transform(dev_headlines) # Se vectorizan los textos de dev
prediction = svc.predict(dev_features) # Se realiza la prediccion

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, prediction, average='macro')*100, 2))}\n")  # Se calcula el F1
print(classification_report(dev_clickbait, prediction))

F1-Score macro: 79.06

              precision    recall  f1-score   support

   Clickbait       0.77      0.63      0.69       203
          No       0.86      0.92      0.89       497

    accuracy                           0.84       700
   macro avg       0.81      0.78      0.79       700
weighted avg       0.83      0.84      0.83       700



TF-IDF()

GaussianNB(var_smoothing=2)

In [ ]:
from sklearn.naive_bayes import GaussianNB

NB = GaussianNB(var_smoothing=3)

training_features = TF_IDF_vectorizer.fit_transform(train_headlines).toarray()
NB.fit(training_features, train_clickbait) # Se entrena el clasificador usando los textos vectorizados

dev_features = TF_IDF_vectorizer.transform(dev_headlines).toarray()
prediction = NB.predict(dev_features) # Se realiza la prediccion

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, prediction, average='macro')*100, 2))}\n")  # Se calcula el F1
print(classification_report(dev_clickbait, prediction))

F1-Score macro: 78.04

              precision    recall  f1-score   support

   Clickbait       0.73      0.64      0.68       203
          No       0.86      0.90      0.88       497

    accuracy                           0.83       700
   macro avg       0.79      0.77      0.78       700
weighted avg       0.82      0.83      0.82       700



TF-IDF()

KNeighborsClassifier(n_neighbors=10)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

KNN = KNeighborsClassifier(n_neighbors=10)

training_features = TF_IDF_vectorizer.fit_transform(train_headlines).toarray() # Se vectorizan los textos de train
KNN.fit(training_features, train_clickbait) # Se entrena el clasificador usando los textos vectorizados

dev_features = TF_IDF_vectorizer.transform(dev_headlines).toarray() # Se vectorizan los textos de dev
prediction = KNN.predict(dev_features) # Se realiza la prediccion

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, prediction, average='macro')*100, 2))}\n")  # Se calcula el F1
print(classification_report(dev_clickbait, prediction))

F1-Score macro: 65.47

              precision    recall  f1-score   support

   Clickbait       0.57      0.42      0.48       203
          No       0.79      0.87      0.82       497

    accuracy                           0.74       700
   macro avg       0.68      0.65      0.65       700
weighted avg       0.72      0.74      0.73       700



BagofWords(ngram_range=(1,3), strip_accents= 'unicode')

LogisticRegression(max_iter=1000, class_weight='balanced')

In [ ]:
bow_vectorizer = CountVectorizer()

training_features = bow_vectorizer.fit_transform(train_headlines) # Se vectorizan los textos de train
LR.fit(training_features, train_clickbait) # Se entrena el clasificador usando los textos vectorizados

dev_features = bow_vectorizer.transform(dev_headlines) # Se vectorizan los textos de dev
prediction = LR.predict(dev_features) # Se realiza la prediccion
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, prediction, average='macro')*100, 2))}\n")  # Se calcula el F1
print(classification_report(dev_clickbait, prediction))

F1-Score macro: 79.4

              precision    recall  f1-score   support

   Clickbait       0.74      0.67      0.70       203
          No       0.87      0.91      0.89       497

    accuracy                           0.84       700
   macro avg       0.81      0.79      0.79       700
weighted avg       0.83      0.84      0.83       700



BagofWords(ngram_range=(1,3), strip_accents= 'unicode')

SVC(C=0.2, kernel='linear')

In [ ]:
bow_vectorizer = CountVectorizer()

svc = SVC(C=0.2, kernel='linear')

training_features = bow_vectorizer.fit_transform(train_headlines) # Se vectorizan los textos de train
svc.fit(training_features, train_clickbait) # Se entrena el clasificador usando los textos vectorizados

dev_features = bow_vectorizer.transform(dev_headlines) # Se vectorizan los textos de dev
prediction = svc.predict(dev_features) # Se realiza la prediccion

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, prediction, average='macro')*100, 2))}\n")  # Se calcula el F1
print(classification_report(dev_clickbait, prediction))

F1-Score macro: 80.4

              precision    recall  f1-score   support

   Clickbait       0.81      0.63      0.71       203
          No       0.86      0.94      0.90       497

    accuracy                           0.85       700
   macro avg       0.84      0.79      0.80       700
weighted avg       0.85      0.85      0.84       700



BagofWords(ngram_range=(1,3), strip_accents= 'unicode')

GaussianNB()

In [ ]:
training_features = bow_vectorizer.fit_transform(train_headlines).toarray()
NB.fit(training_features, train_clickbait) # Se entrena el clasificador usando los textos vectorizados

dev_features = bow_vectorizer.transform(dev_headlines).toarray()
prediction = NB.predict(dev_features) # Se predice si cada texto (ya vectorizado en la línea anterior) es o no clickbait

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, prediction, average='macro')*100, 2))}\n")  # Se calcula el F1
print(classification_report(dev_clickbait, prediction))

F1-Score macro: 41.52

              precision    recall  f1-score   support

   Clickbait       0.00      0.00      0.00       203
          No       0.71      1.00      0.83       497

    accuracy                           0.71       700
   macro avg       0.35      0.50      0.42       700
weighted avg       0.50      0.71      0.59       700



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


BagofWords(ngram_range=(1,3), strip_accents= 'unicode')

KNeighborsClassifier(n_neighbors=6)

In [ ]:
KNN = KNeighborsClassifier(n_neighbors=6)

training_features = bow_vectorizer.fit_transform(train_headlines).toarray()
KNN.fit(training_features, train_clickbait) # Se entrena el clasificador usando los textos vectorizados

dev_features = bow_vectorizer.transform(dev_headlines).toarray()
prediction = KNN.predict(dev_features) # Se predice si cada texto (ya vectorizado en la línea anterior) es o no clickbait

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, prediction, average='macro')*100, 2))}\n")  # Se calcula el F1 y se pasa a escala de 100 para mayor legibilidad.
print(classification_report(dev_clickbait, prediction)) # Se imprime un reporte de clasificación ya implementado en sklearn que, entre otras cosas, contiene al valor de F1 impreso en la línea anterior

F1-Score macro: 57.9

              precision    recall  f1-score   support

   Clickbait       0.41      0.37      0.39       203
          No       0.75      0.78      0.77       497

    accuracy                           0.66       700
   macro avg       0.58      0.58      0.58       700
weighted avg       0.65      0.66      0.66       700



##2️⃣Clasificación con modelos estadísticos usando embeddings como entrada



En esta parte del obligatorio, continuaremos utilizando modelos estadísticos, pero sí comenzaremos a utilizar embeddings.

In [ ]:
%%capture
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import KNeighborsClassifier

embeddings_e5 = SentenceTransformer("intfloat/multilingual-e5-large") # Se carga el modelo de embeddings e5
train_headlines_emb = embeddings_e5.encode(train_headlines) # Se representan los titulares del conjunto de train como embeddings
dev_headlines_emb = embeddings_e5.encode(dev_headlines) # Se representan los titulares del conjunto de dev como embeddings

embedding: SentenceTransformer multilingual-e5-large

clasificador: SVC(C=10, kernel='poly')

In [ ]:
from sklearn.metrics import f1_score, classification_report
from sklearn.svm import SVC

svc = SVC(C=10, kernel='poly')

svc.fit(train_headlines_emb, train_clickbait) # Se ajusta el modelo para tomar como entrada los titulares de train y como salida los valores de clickbait de train.

predictions_svc = svc.predict(dev_headlines_emb) # Una vez ajustado en los valores de train, se predicen nuevas instancias del conjunto de dev.

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_svc, average='macro')*100, 2))}\n") # Se calcula F1.
print(classification_report(dev_clickbait, predictions_svc)) # Se hace el reporte de clasificación de sklearn.

F1-Score macro: 86.84

              precision    recall  f1-score   support

   Clickbait       0.87      0.76      0.81       203
          No       0.91      0.95      0.93       497

    accuracy                           0.90       700
   macro avg       0.89      0.86      0.87       700
weighted avg       0.89      0.90      0.89       700



embedding: SentenceTransformer multilingual-e5-large

clasificador: LogisticRegression(max_iter=1000, class_weight='balanced', C=20)

In [ ]:
from sklearn.linear_model import LogisticRegression

LR = LogisticRegression(max_iter=1000, class_weight='balanced', C=20)

LR.fit(train_headlines_emb, train_clickbait) # Se ajusta el modelo para tomar como entrada los titulares de train y como salida los valores de clickbait de train.

predictions_LR = LR.predict(dev_headlines_emb) # Una vez ajustado en los valores de train, se predicen nuevas instancias del conjunto de dev.

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_LR, average='macro')*100, 2))}\n") # Se calcula F1.
print(classification_report(dev_clickbait, predictions_LR)) # Se hace el reporte de clasificación de sklearn.

F1-Score macro: 86.89

              precision    recall  f1-score   support

   Clickbait       0.79      0.84      0.82       203
          No       0.93      0.91      0.92       497

    accuracy                           0.89       700
   macro avg       0.86      0.88      0.87       700
weighted avg       0.89      0.89      0.89       700



embedding: SentenceTransformer multilingual-e5-large

clasificador: KNeighborsClassifier(n_neighbors=6)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=6)

knn.fit(train_headlines_emb, train_clickbait) # Se ajusta el modelo para tomar como entrada los titulares de train y como salida los valores de clickbait de train.

predictions_knn = knn.predict(dev_headlines_emb) # Una vez ajustado en los valores de train, se predicen nuevas instancias del conjunto de dev.

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_knn, average='macro')*100, 2))}\n") # Se calcula F1.
print(classification_report(dev_clickbait, predictions_knn)) # Se hace el reporte de clasificación de sklearn.

F1-Score macro: 75.82

              precision    recall  f1-score   support

   Clickbait       0.69      0.62      0.65       203
          No       0.85      0.89      0.87       497

    accuracy                           0.81       700
   macro avg       0.77      0.75      0.76       700
weighted avg       0.80      0.81      0.80       700



embedding: SentenceTransformer multilingual-e5-large

clasificador: GaussianNB(var_smoothing=0.1)

In [ ]:
from sklearn.naive_bayes import GaussianNB

NB = GaussianNB(var_smoothing=0.1)

NB.fit(train_headlines_emb, train_clickbait) # Se entrena el clasificador usando los textos vectorizados

prediction = NB.predict(dev_headlines_emb) # Se predice si cada texto (ya vectorizado en la línea anterior) es o no clickbait

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, prediction, average='macro')*100, 2))}\n")  # Se calcula el F1 y se pasa a escala de 100 para mayor legibilidad.
print(classification_report(dev_clickbait, prediction)) # Se imprime un reporte de clasificación ya implementado en sklearn que, entre otras cosas, contiene al valor de F1 impreso en la línea anterior

F1-Score macro: 79.53

              precision    recall  f1-score   support

   Clickbait       0.68      0.76      0.72       203
          No       0.90      0.85      0.87       497

    accuracy                           0.83       700
   macro avg       0.79      0.81      0.80       700
weighted avg       0.83      0.83      0.83       700



embedding: SentenceTransformer Qwen3-Embedding-0.6B

clasificador: KNeighborsClassifier(n_neighbors=4)

In [ ]:
from sentence_transformers import SentenceTransformer

embeddings_qwen3 = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

train_headlines_emb = embeddings_qwen3.encode(train_headlines) # Se representan los titulares del conjunto de train como embeddings
dev_headlines_emb = embeddings_qwen3.encode(dev_headlines) # Se representan los titulares del conjunto de dev como embeddings

In [ ]:
knn = KNeighborsClassifier(n_neighbors=4)

knn.fit(train_headlines_emb, train_clickbait) # Se ajusta el modelo para tomar como entrada los titulares de train y como salida los valores de clickbait de train.

predictions_knn = knn.predict(dev_headlines_emb) # Una vez ajustado en los valores de train, se predicen nuevas instancias del conjunto de dev.

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_knn, average='macro')*100, 2))}\n") # Se calcula F1.
print(classification_report(dev_clickbait, predictions_knn)) # Se hace el reporte de clasificación de sklearn.

F1-Score macro: 64.86

              precision    recall  f1-score   support

   Clickbait       0.50      0.50      0.50       203
          No       0.80      0.80      0.80       497

    accuracy                           0.71       700
   macro avg       0.65      0.65      0.65       700
weighted avg       0.71      0.71      0.71       700



embedding: SentenceTransformer Qwen3-Embedding-0.6B

clasificador: LogisticRegression(max_iter=1000, class_weight='balanced')

In [ ]:
from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression

LR = LogisticRegression(max_iter=1000, class_weight='balanced', C=10)

LR.fit(train_headlines_emb, train_clickbait) # Se ajusta el modelo para tomar como entrada los titulares de train y como salida los valores de clickbait de train.

predictions_LR = LR.predict(dev_headlines_emb) # Una vez ajustado en los valores de train, se predicen nuevas instancias del conjunto de dev.

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_LR, average='macro')*100, 2))}\n") # Se calcula F1.
print(classification_report(dev_clickbait, predictions_LR)) # Se hace el reporte de clasificación de sklearn.

F1-Score macro: 77.29

              precision    recall  f1-score   support

   Clickbait       0.65      0.73      0.69       203
          No       0.88      0.84      0.86       497

    accuracy                           0.81       700
   macro avg       0.76      0.78      0.77       700
weighted avg       0.82      0.81      0.81       700



embedding: SentenceTransformer Qwen3-Embedding-0.6B

clasificador: SVC(C=6, kernel='linear')

In [ ]:
from sklearn.svm import SVC

svc = SVC(C=6, kernel='linear')

svc.fit(train_headlines_emb, train_clickbait) # Se ajusta el modelo para tomar como entrada los titulares de train y como salida los valores de clickbait de train.

predictions_svc = svc.predict(dev_headlines_emb) # Una vez ajustado en los valores de train, se predicen nuevas instancias del conjunto de dev.

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_svc, average='macro')*100, 2))}\n") # Se calcula F1.
print(classification_report(dev_clickbait, predictions_svc)) # Se hace el reporte de clasificación de sklearn.

F1-Score macro: 78.99

              precision    recall  f1-score   support

   Clickbait       0.76      0.64      0.69       203
          No       0.86      0.92      0.89       497

    accuracy                           0.84       700
   macro avg       0.81      0.78      0.79       700
weighted avg       0.83      0.84      0.83       700



embedding: SentenceTransformer Qwen3-Embedding-0.6B

clasificador: GaussianNB(var_smoothing=0.1)


In [ ]:
from sklearn.naive_bayes import GaussianNB

NB = GaussianNB(var_smoothing=0.1)

NB.fit(train_headlines_emb, train_clickbait) # Se entrena el clasificador usando los textos vectorizados

prediction = NB.predict(dev_headlines_emb) # Se predice si cada texto (ya vectorizado en la línea anterior) es o no clickbait

print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, prediction, average='macro')*100, 2))}\n")  # Se calcula el F1 y se pasa a escala de 100 para mayor legibilidad.
print(classification_report(dev_clickbait, prediction)) # Se imprime un reporte de clasificación ya implementado en sklearn que, entre otras cosas, contiene al valor de F1 impreso en la línea anterior

F1-Score macro: 70.87

              precision    recall  f1-score   support

   Clickbait       0.56      0.64      0.60       203
          No       0.84      0.80      0.82       497

    accuracy                           0.75       700
   macro avg       0.70      0.72      0.71       700
weighted avg       0.76      0.75      0.76       700



##3️⃣Clasificación con LLMs usando zero-shot

En esta parte del laboratorio, usaremos LLMs (Large Language Models) para predecir si un titular dado es clickbait o no. Al utilizar Few-Shot, no incluiremos ejemplos de clickbait en las prompts. Utilizaremos tres LLMs distintos:


*   Qwen/Qwen3-4B-Instruct-2507
*   ibm-granite/granite-4.0-h-350M
*   Llama 3.2-3B

Además, para cada LLM, probaremos tanto indicarle una definición de clickbait como no, y evaluaremos las diferencias en los resultados obtenidos.





La definición de clickbait que utilizaremos fue escrita por nosotros y busca hacer uso del hecho que no estamos preprocesando los titulares de ninguna forma. Creemos que es posible que un preprocesamiento muy transformativo de los titulares pude disminuir el rendimiento del desempeño, porque, justamente, para clasificar un titular como Clickbait, el sobreuso de puntuación, mayúsculas, emoticones, entre otros son factores pertinentes. En este sentido, la prompt busca darle importancia a estos elementos para ayudar a los LLM a clasificar un titular, sin que estos factores se vuelvan intrínsecos en si una noticia es clickbait o no.

In [ ]:
%%capture
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_name) # Se carga el tokenizador asociado al modelo elegido
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto") # Se carga el modelo desde hugging face

modelo: Qwen/Qwen3-4B-Instruct-2507

Definición: No


In [ ]:
from sklearn.metrics import f1_score, classification_report
import torch
import math

print("Iniciando predicciones con LLM (Zero-Shot) sobre todo el conjunto DEV...")

def crear_prompt(texto):
    prompt = f'El siguiente es un titular de noticia: "{texto}".\n¿Es este titular clickbait? Responde únicamente "Clickbait" o "No".'
    messages = [{"role": "user", "content": prompt}]
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text


print(f"Preparando {len(dev_headlines)} prompts...")
all_formatted_prompts = [crear_prompt(texto) for texto in dev_headlines]

# se define un batch para procesar al mismo tiempo
batch_size = 16
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")


with torch.no_grad():
    # Iteramos sobre la lista de prompts en varios batches del tamanio batch_size
    for i in range(0, len(all_formatted_prompts), batch_size):

        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")
        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip()

            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"

            all_predictions_clean.append(clean_pred)

print("\nProceso completado. Evaluando resultados...")

predictions_clean = all_predictions_clean

print(f"\nResultados del LLM: {model_name}\n")
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(dev_clickbait, predictions_clean))

# (mostramos algunos ejemplos)
print("\n--- Ejemplos de Predicciones ---")
for i in range(min(10, len(dev_headlines))):
    print(f"Texto: {dev_headlines[i]}")
    print(f"  -> Predicción: '{predictions_clean[i]}'")
    print(f"  -> Esperado:   '{dev_clickbait[i]}'")

Iniciando predicciones con LLM (Zero-Shot) sobre todo el conjunto DEV...
Preparando 700 prompts...
Iniciando procesamiento en 44 lotes (batch_size=16)...
  -> Procesando lote 1 de 44...
  -> Procesando lote 2 de 44...
  -> Procesando lote 3 de 44...
  -> Procesando lote 4 de 44...
  -> Procesando lote 5 de 44...
  -> Procesando lote 6 de 44...
  -> Procesando lote 7 de 44...
  -> Procesando lote 8 de 44...
  -> Procesando lote 9 de 44...
  -> Procesando lote 10 de 44...
  -> Procesando lote 11 de 44...
  -> Procesando lote 12 de 44...
  -> Procesando lote 13 de 44...
  -> Procesando lote 14 de 44...
  -> Procesando lote 15 de 44...
  -> Procesando lote 16 de 44...
  -> Procesando lote 17 de 44...
  -> Procesando lote 18 de 44...
  -> Procesando lote 19 de 44...
  -> Procesando lote 20 de 44...
  -> Procesando lote 21 de 44...
  -> Procesando lote 22 de 44...
  -> Procesando lote 23 de 44...
  -> Procesando lote 24 de 44...
  -> Procesando lote 25 de 44...
  -> Procesando lote 26 de 44.

modelo: Qwen/Qwen3-4B-Instruct-2507

Definicion: Sí

In [ ]:
from sklearn.metrics import f1_score, classification_report
import torch
import math

print("Iniciando predicciones con LLM (Zero-Shot) sobre todo el conjunto DEV...")

def crear_prompt(texto):
    prompt = f"""
    Clickbait es una noticia que omite deliberadamente o tergiversa informacion con el objetivo de resultar mas atractiva y atraer clicks.
    Suelen hacer uso de puntuacion exagerada, uso de emojis y terminos con demasiada connotacion positiva o negativa, y tienen titulos sensacionales.”

    Indique si el siguiente titular es Clickbait o no:
    "{texto}"

    Responde solamente "Clickbait" o "No".
    """
    messages = [{"role": "user", "content": prompt}]
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text

print(f"Preparando {len(dev_headlines)} prompts...")
all_formatted_prompts = [crear_prompt(texto) for texto in dev_headlines]

batch_size = 16
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):

        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")

        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        #se normalizan las respuestas obtenidas
        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip()

            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"

            all_predictions_clean.append(clean_pred)

print("\nProceso completado. Evaluando resultados...")

predictions_clean = all_predictions_clean

print(f"\nResultados del LLM: {model_name}\n")
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(dev_clickbait, predictions_clean))

# Ver algunas predicciones
print("\n--- Ejemplos de Predicciones ---")
for i in range(min(10, len(dev_headlines))):
    print(f"Texto: {dev_headlines[i]}")
    print(f"  -> Predicción: '{predictions_clean[i]}'")
    print(f"  -> Esperado:   '{dev_clickbait[i]}'")

Iniciando predicciones con LLM (Zero-Shot) sobre todo el conjunto DEV...
Preparando 700 prompts...
Iniciando procesamiento en 44 lotes (batch_size=16)...
  -> Procesando lote 1 de 44...
  -> Procesando lote 2 de 44...
  -> Procesando lote 3 de 44...
  -> Procesando lote 4 de 44...
  -> Procesando lote 5 de 44...
  -> Procesando lote 6 de 44...
  -> Procesando lote 7 de 44...
  -> Procesando lote 8 de 44...
  -> Procesando lote 9 de 44...
  -> Procesando lote 10 de 44...
  -> Procesando lote 11 de 44...
  -> Procesando lote 12 de 44...
  -> Procesando lote 13 de 44...
  -> Procesando lote 14 de 44...
  -> Procesando lote 15 de 44...
  -> Procesando lote 16 de 44...
  -> Procesando lote 17 de 44...
  -> Procesando lote 18 de 44...
  -> Procesando lote 19 de 44...
  -> Procesando lote 20 de 44...
  -> Procesando lote 21 de 44...
  -> Procesando lote 22 de 44...
  -> Procesando lote 23 de 44...
  -> Procesando lote 24 de 44...
  -> Procesando lote 25 de 44...
  -> Procesando lote 26 de 44.

En este primer caso, el desempeño bajó al incluir la definición. Notamos en los ejemplos impresos que ninguno hace uso de los recursos explicitados en la definición dada. Esto puede apuntar a que la definición utilizada no es particularmente atinada, o al menos no en este caso. Además, al incluir la definición notamos un recall de la clase "Clickbait" muy bajo, lo que indica que la definición provocó que el modelo se vuelva mucho más cauteloso al clasificar un titular como Clickbait.

Modelo: Llama 3.2 - 3B.


Definicion: no


Para usar este LLM, se precisa una key. En caso de querer usarlo, hay que iniciar sesion en la proxima celda. La key esta en Secrets del lab.

In [4]:
from huggingface_hub import login
login()

In [5]:
%%capture
!pip install --upgrade transformers

import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)


In [ ]:
from sklearn.metrics import f1_score, classification_report


predicciones = []
index_dev = 0
for headline in dev_headlines:
  text = dev_headlines[index_dev]


  prompt = f'El siguiente es un titular de noticia: "{text}".\n¿Es este titular clickbait? Responde únicamente "Clickbait" o "No".'

  messages = [{"role": "user", "content": prompt}] # Esto tiene que ver con el formato final de prompt que le va a entrar al modelo, con el cual fue ajustado.
  text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) # Esto también tiene que ver con el formato final de prompt que le va a entrar al modelo.
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device) # Se tokeniza la entrada

  generated_ids = model.generate(**model_inputs, max_new_tokens=16384) # El modelo genera de acuerdo a la entrada ya tokenizada
  output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() # Se guarda solamente lo que se necesita de la salida del modelo
  content = tokenizer.decode(output_ids, skip_special_tokens=True) # Se recupera en texto lo generado por el modelo

  if(index_dev % 100 == 0): #para ir viendo cuanto falta je
    print(index_dev, "/700")

  if "clickbait" in content.lower(): #vamos guardando en la lista
        predicciones.append("Clickbait")
  else:
        predicciones.append("No")

  index_dev += 1
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predicciones, average='macro')*100, 2))}\n")  # Se calcula el F1 y se pasa a escala de 100 para mayor legibilidad.
print(classification_report(dev_clickbait, predicciones))

0 /700
100 /700
200 /700
300 /700
400 /700
500 /700
600 /700
F1-Score macro: 55.72

              precision    recall  f1-score   support

   Clickbait       0.38      0.87      0.53       203
          No       0.89      0.43      0.58       497

    accuracy                           0.56       700
   macro avg       0.64      0.65      0.56       700
weighted avg       0.74      0.56      0.57       700



Modelo: Modelo: Llama 3.2 - 3B.

Definicion: si



In [ ]:
from sklearn.metrics import f1_score, classification_report


predicciones = []
index_dev = 0
for headline in dev_headlines:
  text = dev_headlines[index_dev]

#definicion de https://www.fundeu.es/noticia/ciberanzuelo-la-alternativa-favorita-de-nuestros-seguidores-a-clickbait
  prompt = f"""
  Clickbait es una noticia que omite deliberadamente o tergiversa informacion con el objetivo de resultar mas atractiva y atraer clicks.
  Suelen hacer uso de puntuacion exagerada, uso de emojis y terminos con demasiada connotacion positiva o negativa, y tienen titulos sensacionales.”

  Indique si el siguiente titular es Clickbait o no:
  "{text}"

  Responde solamente "Clickbait" o "No".
  """

  messages = [{"role": "user", "content": prompt}] # Esto tiene que ver con el formato final de prompt que le va a entrar al modelo, con el cual fue ajustado.
  text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) # Esto también tiene que ver con el formato final de prompt que le va a entrar al modelo.
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device) # Se tokeniza la entrada

  generated_ids = model.generate(**model_inputs, max_new_tokens=16384) # El modelo genera de acuerdo a la entrada ya tokenizada
  output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() # Se guarda solamente lo que se necesita de la salida del modelo
  content = tokenizer.decode(output_ids, skip_special_tokens=True) # Se recupera en texto lo generado por el modelo

  #print("Prediccion:", content)
  #print("Real:", dev_clickbait[index_dev])
  if(index_dev % 100 == 0): #para ir viendo cuanto falta je
    print(index_dev, "/700")

  if "clickbait" in content.lower(): #vamos guardando en la lista
        predicciones.append("Clickbait")
  else:
        predicciones.append("No")

  index_dev += 1
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predicciones, average='macro')*100, 2))}\n")  # Se calcula el F1 y se pasa a escala de 100 para mayor legibilidad.
print(classification_report(dev_clickbait, predicciones))

0 /700
100 /700
200 /700
300 /700
400 /700
500 /700
600 /700
F1-Score macro: 60.5

              precision    recall  f1-score   support

   Clickbait       0.43      0.48      0.45       203
          No       0.78      0.74      0.76       497

    accuracy                           0.66       700
   macro avg       0.60      0.61      0.60       700
weighted avg       0.68      0.66      0.67       700



Notamos que, al incluir esta definición, aumentaron tanto la precisión de la categoría Clickbait, como el recall de la categoría No Clickbait. Es decir: cuando predice que una noticia es Clickbait, se equivoca menos (aunque aún se equivoca bastante). Además, como hay menos "falsos positivos" y más "falsos negativos", nuevamente concluimos que con nuestra definición el modelo se volvió más "estricto" con qué es clickbait y qué no. En este caso, esto hizo que aumente el desempeño (en términos de F1)

---


modelo utilizado: ibm-granite/granite-4.0-h-350M

Definición: No

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import f1_score, classification_report
import torch
import math

tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-4.0-h-350M")
model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-4.0-h-350M", dtype="auto", device_map="auto")

print("Iniciando predicciones con LLM (Zero-Shot) sobre todo el conjunto DEV...")

def crear_prompt(texto):
    prompt = f'El siguiente es un titular de noticia: "{texto}".\n¿Es este titular clickbait? Responde únicamente "Clickbait" o "No".'
    messages = [{"role": "user", "content": prompt}]
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text

print(f"Preparando {len(dev_headlines)} prompts...")
all_formatted_prompts = [crear_prompt(texto) for texto in dev_headlines]

batch_size = 4
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):

        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")

        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip()

            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"

            all_predictions_clean.append(clean_pred)

print("\nProceso completado. Evaluando resultados...")

predictions_clean = all_predictions_clean

print(f"\nResultados del LLM: ibm-granite/granite-4.0-h-350M\n")
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(dev_clickbait, predictions_clean))

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.15M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/6.42k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  681MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/370 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Iniciando predicciones con LLM (Zero-Shot) sobre todo el conjunto DEV...
Preparando 700 prompts...
Iniciando procesamiento en 175 lotes (batch_size=4)...
  -> Procesando lote 1 de 175...
  -> Procesando lote 2 de 175...
  -> Procesando lote 3 de 175...
  -> Procesando lote 4 de 175...
  -> Procesando lote 5 de 175...
  -> Procesando lote 6 de 175...
  -> Procesando lote 7 de 175...
  -> Procesando lote 8 de 175...
  -> Procesando lote 9 de 175...
  -> Procesando lote 10 de 175...
  -> Procesando lote 11 de 175...
  -> Procesando lote 12 de 175...
  -> Procesando lote 13 de 175...
  -> Procesando lote 14 de 175...
  -> Procesando lote 15 de 175...
  -> Procesando lote 16 de 175...
  -> Procesando lote 17 de 175...
  -> Procesando lote 18 de 175...
  -> Procesando lote 19 de 175...
  -> Procesando lote 20 de 175...
  -> Procesando lote 21 de 175...
  -> Procesando lote 22 de 175...
  -> Procesando lote 23 de 175...
  -> Procesando lote 24 de 175...
  -> Procesando lote 25 de 175...
  -> 

Este caso tiene la particularidad de que el LLM predijo el 100% de los casos como Clickbait. Es claro que la tarea de clasificación no fue cumplida de forma satisfactoria en este caso, aunque es posible que mejore el desempeño al cambiar la prompt.

---

modelo: ibm-granite/granite-4.0-h-350M

Definición: sí


In [ ]:
import math

print("Iniciando predicciones con LLM (Zero-Shot) sobre todo el conjunto DEV...")

def crear_prompt(texto):
    prompt = f"""
    Clickbait es una noticia que omite deliberadamente o tergiversa informacion con el objetivo de resultar mas atractiva y atraer clicks.
    Suelen hacer uso de puntuacion exagerada, uso de emojis y terminos con demasiada connotacion positiva o negativa, y tienen titulos sensacionales.”

    Indique si el siguiente titular es Clickbait o no:
    "{texto}"

    Responde solamente "Clickbait" o "No".
    """
    messages = [{"role": "user", "content": prompt}]
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text

print(f"Preparando {len(dev_headlines)} prompts...")
all_formatted_prompts = [crear_prompt(texto) for texto in dev_headlines]

batch_size = 4
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):

        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")

        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip()

            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"

            all_predictions_clean.append(clean_pred)

print("\nProceso completado. Evaluando resultados...")

predictions_clean = all_predictions_clean

print(f"\nResultados del LLM: ibm-granite/granite-4.0-h-350M\n")
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(dev_clickbait, predictions_clean))

Iniciando predicciones con LLM (Zero-Shot) sobre todo el conjunto DEV...
Preparando 700 prompts...
Iniciando procesamiento en 175 lotes (batch_size=4)...
  -> Procesando lote 1 de 175...
  -> Procesando lote 2 de 175...
  -> Procesando lote 3 de 175...
  -> Procesando lote 4 de 175...
  -> Procesando lote 5 de 175...
  -> Procesando lote 6 de 175...
  -> Procesando lote 7 de 175...
  -> Procesando lote 8 de 175...
  -> Procesando lote 9 de 175...
  -> Procesando lote 10 de 175...
  -> Procesando lote 11 de 175...
  -> Procesando lote 12 de 175...
  -> Procesando lote 13 de 175...
  -> Procesando lote 14 de 175...
  -> Procesando lote 15 de 175...
  -> Procesando lote 16 de 175...
  -> Procesando lote 17 de 175...
  -> Procesando lote 18 de 175...
  -> Procesando lote 19 de 175...
  -> Procesando lote 20 de 175...
  -> Procesando lote 21 de 175...
  -> Procesando lote 22 de 175...
  -> Procesando lote 23 de 175...
  -> Procesando lote 24 de 175...
  -> Procesando lote 25 de 175...
  -> 

Hubo una variación muy grande al incluir la definición de clickbait. En particular, parece haber sucedido una clasificación de alguna forma inversa a la del caso anterior, pues ahora el recall de la clase Clickbait es muy bajo y el de la clase "No" es alto. Es decir, que se predijeron muchos titulares como "No clickbait".

##4️⃣Clasificación con LLMs usando few-shot

En esta parte del laboratorio, probamos utilizar LLMs pero con Few-Shot, es decir, incluyendo algunos ejemplos en la prompt. En particular, probamos variamos la cantidad de ejemplos elegidos, la forma de elegirlos (de forma aleatoria o dinamica) y la inclusion o no de la prompt de las partes anteriores. La combinacion de estas variables genera varios posibles casos. Sin embargo, mantendremos fijos algunos parametros para poder extraer conclusiones más verosímiles (por ejemplo, un modelo incluye la definición en la prompt sí y solo sí incluye 5 ejemplos).

### Few-shot con ejemplos elegidos de manera manual o aleatoria

Para mantener continuidad, en los casos en los que usamos la definición, es la misma que usamos en la parte 3. En lo que sigue, utilizamos dos variaciones distintas para cada LLM:

*   Con 2 ejemplos elegidos aleatoriamente y sin definición incluida en la prompt.
*   Con 5 ejemplos aleatoriamente y con definición incluida en la prompt.



modelo: Qwen/Qwen3-4B-Instruct-2507

prompt: Sin definición

m = 2 elegidos aleatoriamente

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import f1_score, classification_report
import torch
import random
import math
import gc

model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')

model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")

print("Iniciando predicciones con LLM (Few-Shot Manual) sobre todo el conjunto dev...")


def crear_prompt_fewshot(texto_nuevo):
    # Esta es la instrucción principal
    instruccion_prompt = 'Indicá si este titular es clickbait o no. Solamente respondé "Clickbait" o "No".\nAquí tienes algunos ejemplos:'

    # Usamos el formato de chat para "enseñarle" al modelo
    random_indices = random.sample(range(len(train_headlines)), k=2)
    messages = [
        # 1. La instrucción
        {"role": "user", "content": instruccion_prompt},

        # 2. Ejemplo 1 (Usuario pregunta, Asistente responde)
        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[0]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[0]]},

        # 3. Ejemplo 2
        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[1]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[1]]},

        # 4. La nueva pregunta que debe responder
        {"role": "user", "content": f'Titular: "{texto_nuevo}"'}
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text


print(f"Preparando {len(dev_headlines)} prompts...")
all_formatted_prompts = [crear_prompt_fewshot(texto) for texto in dev_headlines]


batch_size = 16
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):

        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")

        # 1. Lote actual
        batch_prompts = all_formatted_prompts[i : i + batch_size]

        # 2. Tokenizar
        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        # 3. Generar
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        # 4. Decodificar
        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        # 5. Limpiar y guardar
        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip().replace(".","")

            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"

            all_predictions_clean.append(clean_pred)

        # 6. Forzar Limpieza de Memoria
        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()


print("\nProceso completado. Evaluando resultados...")

predictions_clean = all_predictions_clean

# Usamos dev_clickbait
print(f"\nResultados del LLM (Few-Shot Manual, m=2): {model_name}\n")
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(dev_clickbait, predictions_clean))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Iniciando predicciones con LLM (Few-Shot Manual) sobre todo el conjunto dev...
Preparando 700 prompts...
Iniciando procesamiento en 44 lotes (batch_size=16)...
  -> Procesando lote 1 de 44...
  -> Procesando lote 2 de 44...
  -> Procesando lote 3 de 44...
  -> Procesando lote 4 de 44...
  -> Procesando lote 5 de 44...
  -> Procesando lote 6 de 44...
  -> Procesando lote 7 de 44...
  -> Procesando lote 8 de 44...
  -> Procesando lote 9 de 44...
  -> Procesando lote 10 de 44...
  -> Procesando lote 11 de 44...
  -> Procesando lote 12 de 44...
  -> Procesando lote 13 de 44...
  -> Procesando lote 14 de 44...
  -> Procesando lote 15 de 44...
  -> Procesando lote 16 de 44...
  -> Procesando lote 17 de 44...
  -> Procesando lote 18 de 44...
  -> Procesando lote 19 de 44...
  -> Procesando lote 20 de 44...
  -> Procesando lote 21 de 44...
  -> Procesando lote 22 de 44...
  -> Procesando lote 23 de 44...
  -> Procesando lote 24 de 44...
  -> Procesando lote 25 de 44...
  -> Procesando lote 26 

modelo: Qwen/Qwen3-4B-Instruct-2507

prompt: Con definición

m = 5 elegidos aleatoriamente

In [ ]:
import random
import gc

print("Iniciando predicciones con LLM (Few-Shot Manual) sobre todo el conjunto dev...")

def crear_prompt_fewshot(texto_nuevo):
    instruccion_prompt = f"""
    Clickbait es una noticia o articulo que omite deliberadamente o tergiversa informacion con el objetivo de resultar mas atractiva y atraer clicks.
    Suelen hacer uso de puntuacion exagerada, uso de emojis y terminos con demasiada connotacion positiva o negativa, y tienen titulos sensacionales.”


    Indique si el siguiente titular es Clickbait o no:
    "{texto_nuevo}"


    Responde solamente "Clickbait" o "No".
    Aqui tienes algunos ejemplos:
    """


    random_indices = random.sample(range(len(train_headlines)), k=5)
    messages = [
        {"role": "user", "content": instruccion_prompt},

        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[0]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[0]]},

        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[1]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[1]]},

        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[2]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[2]]},

        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[3]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[3]]},

        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[4]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[4]]},

        {"role": "user", "content": f'Titular: "{texto_nuevo}"'}
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text

print(f"Preparando {len(dev_headlines)} prompts...")
all_formatted_prompts = [crear_prompt_fewshot(texto) for texto in dev_headlines]

batch_size = 16
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):

        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")

        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip().replace(".","")

            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"

            all_predictions_clean.append(clean_pred)

        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()

print("\nProceso completado. Evaluando resultados...")

predictions_clean = all_predictions_clean

print(f"\nResultados del LLM (Few-Shot Manual, m=5): {model_name}\n")
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(dev_clickbait, predictions_clean))

Iniciando predicciones con LLM (Few-Shot Manual) sobre todo el conjunto dev...
Preparando 700 prompts...
Iniciando procesamiento en 44 lotes (batch_size=16)...
  -> Procesando lote 1 de 44...
  -> Procesando lote 2 de 44...
  -> Procesando lote 3 de 44...
  -> Procesando lote 4 de 44...
  -> Procesando lote 5 de 44...
  -> Procesando lote 6 de 44...
  -> Procesando lote 7 de 44...
  -> Procesando lote 8 de 44...
  -> Procesando lote 9 de 44...
  -> Procesando lote 10 de 44...
  -> Procesando lote 11 de 44...
  -> Procesando lote 12 de 44...
  -> Procesando lote 13 de 44...
  -> Procesando lote 14 de 44...
  -> Procesando lote 15 de 44...
  -> Procesando lote 16 de 44...
  -> Procesando lote 17 de 44...
  -> Procesando lote 18 de 44...
  -> Procesando lote 19 de 44...
  -> Procesando lote 20 de 44...
  -> Procesando lote 21 de 44...
  -> Procesando lote 22 de 44...
  -> Procesando lote 23 de 44...
  -> Procesando lote 24 de 44...
  -> Procesando lote 25 de 44...
  -> Procesando lote 26 

modelo: ibm-granite/granite-4.0-h-350M

prompt: Sin definición

m = 2 elegidos aleatoriamente

In [ ]:
import random
from sklearn.metrics import f1_score, classification_report
import torch
import math
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "ibm-granite/granite-4.0-h-350M"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")

print("Iniciando predicciones con LLM (Few-Shot Manual) sobre todo el conjunto dev...")

def crear_prompt_fewshot(texto_nuevo):
    instruccion_prompt = 'Indicá si este titular es clickbait o no. Solamente respondé "Clickbait" o "No".\nAquí tienes algunos ejemplos:'

    random_indices = random.sample(range(len(train_headlines)), k=2)
    messages = [
        {"role": "user", "content": instruccion_prompt},

        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[0]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[0]]},

        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[1]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[1]]},

        {"role": "user", "content": f'Titular: "{texto_nuevo}"'}
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text

print(f"Preparando {len(dev_headlines)} prompts...")
all_formatted_prompts = [crear_prompt_fewshot(texto) for texto in dev_headlines]

batch_size = 2
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):

        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")
        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip().replace(".","")

            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"

            all_predictions_clean.append(clean_pred)

        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()

print("\nProceso completado. Evaluando resultados...")

predictions_clean = all_predictions_clean

print(f"\nResultados del LLM (Few-Shot Manual, m=2): {model_name}\n")
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(dev_clickbait, predictions_clean))

Iniciando predicciones con LLM (Few-Shot Manual) sobre todo el conjunto dev...
Preparando 700 prompts...
Iniciando procesamiento en 350 lotes (batch_size=2)...
  -> Procesando lote 1 de 350...
  -> Procesando lote 2 de 350...
  -> Procesando lote 3 de 350...
  -> Procesando lote 4 de 350...
  -> Procesando lote 5 de 350...
  -> Procesando lote 6 de 350...
  -> Procesando lote 7 de 350...
  -> Procesando lote 8 de 350...
  -> Procesando lote 9 de 350...
  -> Procesando lote 10 de 350...
  -> Procesando lote 11 de 350...
  -> Procesando lote 12 de 350...
  -> Procesando lote 13 de 350...
  -> Procesando lote 14 de 350...
  -> Procesando lote 15 de 350...
  -> Procesando lote 16 de 350...
  -> Procesando lote 17 de 350...
  -> Procesando lote 18 de 350...
  -> Procesando lote 19 de 350...
  -> Procesando lote 20 de 350...
  -> Procesando lote 21 de 350...
  -> Procesando lote 22 de 350...
  -> Procesando lote 23 de 350...
  -> Procesando lote 24 de 350...
  -> Procesando lote 25 de 350...

modelo: ibm-granite/granite-4.0-h-350M

prompt: Con definición

m = 5 elegidos aleatoriamente

In [ ]:
import random
from sklearn.metrics import f1_score, classification_report
import torch
import math
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "ibm-granite/granite-4.0-h-350M"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")

print("Iniciando predicciones con LLM (Few-Shot Manual) sobre todo el conjunto DEV...")

def crear_prompt_fewshot(texto_nuevo):
    instruccion_prompt = f"""
    Clickbait es una noticia o articulo que omite deliberadamente o tergiversa informacion con el objetivo de resultar mas atractiva y atraer clicks.
    Suelen hacer uso de puntuacion exagerada, uso de emojis y terminos con demasiada connotacion positiva o negativa, y tienen titulos sensacionales.”


    Indique si el siguiente titular es Clickbait o no:
    "{texto_nuevo}"


    Responde solamente "Clickbait" o "No".
    Aqui tienes algunos ejemplos:
    """

    random_indices = random.sample(range(len(train_headlines)), k=5)
    messages = [

        {"role": "user", "content": instruccion_prompt},

        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[0]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[0]]},

        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[1]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[1]]},

        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[2]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[2]]},

        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[3]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[3]]},

        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[4]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[4]]},

        {"role": "user", "content": f'Titular: "{texto_nuevo}"'}
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text


print(f"Preparando {len(dev_headlines)} prompts...")
all_formatted_prompts = [crear_prompt_fewshot(texto) for texto in dev_headlines]


batch_size = 2
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):

        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")

        batch_prompts = all_formatted_prompts[i : i + batch_size]


        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)


        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)


        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip().replace(".","")

            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"

            all_predictions_clean.append(clean_pred)

        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()

print("\nProceso completado. Evaluando resultados...")

predictions_clean = all_predictions_clean


print(f"\nResultados del LLM (Few-Shot Manual, m=5): {model_name}\n")
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(dev_clickbait, predictions_clean))

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/681M [00:00<?, ?B/s]

The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Iniciando predicciones con LLM (Few-Shot Manual) sobre todo el conjunto DEV...
Preparando 700 prompts...
Iniciando procesamiento en 350 lotes (batch_size=2)...
  -> Procesando lote 1 de 350...
  -> Procesando lote 2 de 350...
  -> Procesando lote 3 de 350...
  -> Procesando lote 4 de 350...
  -> Procesando lote 5 de 350...
  -> Procesando lote 6 de 350...
  -> Procesando lote 7 de 350...
  -> Procesando lote 8 de 350...
  -> Procesando lote 9 de 350...
  -> Procesando lote 10 de 350...
  -> Procesando lote 11 de 350...
  -> Procesando lote 12 de 350...
  -> Procesando lote 13 de 350...
  -> Procesando lote 14 de 350...
  -> Procesando lote 15 de 350...
  -> Procesando lote 16 de 350...
  -> Procesando lote 17 de 350...
  -> Procesando lote 18 de 350...
  -> Procesando lote 19 de 350...
  -> Procesando lote 20 de 350...
  -> Procesando lote 21 de 350...
  -> Procesando lote 22 de 350...
  -> Procesando lote 23 de 350...
  -> Procesando lote 24 de 350...
  -> Procesando lote 25 de 350...

Modelo: Llama 3.2-3B

Prompt: Sin definición

m = 2 (elegidos de forma aleatoria)

In [ ]:
import random
from sklearn.metrics import f1_score, classification_report
import torch
import math
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Iniciando predicciones con LLM (Few-Shot Manual) sobre todo el conjunto dev...")

def crear_prompt_fewshot(texto_nuevo):
    instruccion_prompt = (
        "Indicá si este titular es clickbait o no. "
        "Respondé únicamente con 'Clickbait' o 'No'.\n"
        "Aquí tienes algunos ejemplos:"
    )

    random_indices = random.sample(range(len(train_headlines)), k=2)

    messages = [
        {"role": "system", "content": "Sos un clasificador de titulares."},
        {"role": "user", "content": instruccion_prompt},
        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[0]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[0]]},
        {"role": "user", "content": f'Titular: "{train_headlines[random_indices[1]]}"'},
        {"role": "assistant", "content": train_clickbait[random_indices[1]]},
        {"role": "user", "content": f'Titular: "{texto_nuevo}"'}
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text

print(f"Preparando {len(dev_headlines)} prompts...")
all_formatted_prompts = [crear_prompt_fewshot(texto) for texto in dev_headlines]

batch_size = 2
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):
        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")
        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean = pred.strip().split("\n")[0].strip().replace(".", "")
            if clean not in ["Clickbait", "No"]:
                if "clickbait" in clean.lower():
                    clean = "Clickbait"
                else:
                    clean = "No"
            all_predictions_clean.append(clean)

        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()

print("\nProceso completado. Evaluando resultados...\n")

predictions_clean = all_predictions_clean

print(f"Resultados del LLM (Few-Shot Manual, m=2): {model_name}\n")
print(f"F1-Score macro: {round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2)}\n")
print(classification_report(dev_clickbait, predictions_clean))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Iniciando predicciones con LLM (Few-Shot Manual) sobre todo el conjunto dev...
Preparando 700 prompts...
Iniciando procesamiento en 350 lotes (batch_size=2)...
  -> Procesando lote 1 de 350...
  -> Procesando lote 2 de 350...
  -> Procesando lote 3 de 350...
  -> Procesando lote 4 de 350...
  -> Procesando lote 5 de 350...
  -> Procesando lote 6 de 350...
  -> Procesando lote 7 de 350...
  -> Procesando lote 8 de 350...
  -> Procesando lote 9 de 350...
  -> Procesando lote 10 de 350...
  -> Procesando lote 11 de 350...
  -> Procesando lote 12 de 350...
  -> Procesando lote 13 de 350...
  -> Procesando lote 14 de 350...
  -> Procesando lote 15 de 350...
  -> Procesando lote 16 de 350...
  -> Procesando lote 17 de 350...
  -> Procesando lote 18 de 350...
  -> Procesando lote 19 de 350...
  -> Procesando lote 20 de 350...
  -> Procesando lote 21 de 350...
  -> Procesando lote 22 de 350...
  -> Procesando lote 23 de 350...
  -> Procesando lote 24 de 350...
  -> Procesando lote 25 de 350...

Modelo: Llama 3.2-3B

Prompt: Con definición

m = 5

In [ ]:
import random
from sklearn.metrics import f1_score, classification_report
import torch
import math
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

def crear_prompt_fewshot(texto_nuevo):
    instruccion_prompt = f"""
    Clickbait es una noticia o articulo que omite deliberadamente o tergiversa informacion con el objetivo de resultar mas atractiva y atraer clicks.
    Suelen hacer uso de puntuacion exagerada, uso de emojis y terminos con demasiada connotacion positiva o negativa, y tienen titulos sensacionales.”


    Indique si el siguiente titular es Clickbait o no:
    "{texto_nuevo}"


    Responde solamente "Clickbait" o "No".
    Aqui tienes algunos ejemplos:
    """

    random_indices = random.sample(range(len(train_headlines)), k=5)

    messages = [
        {"role": "system", "content": "Sos un clasificador de titulares."},
        {"role": "user", "content": instruccion_prompt},
    ]

    for idx in random_indices:
        messages.append({"role": "user", "content": f'Titular: "{train_headlines[idx]}"'})
        messages.append({"role": "assistant", "content": train_clickbait[idx]})

    messages.append({"role": "user", "content": f'Titular: "{texto_nuevo}"'})

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

print(f"Preparando {len(dev_headlines)} prompts...")
all_formatted_prompts = [crear_prompt_fewshot(texto) for texto in dev_headlines]

batch_size = 2
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):
        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean = pred.strip().split("\n")[0].strip().replace(".", "")
            if clean not in ["Clickbait", "No"]:
                if "clickbait" in clean.lower():
                    clean = "Clickbait"
                else:
                    clean = "No"
            all_predictions_clean.append(clean)

        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()

predictions_clean = all_predictions_clean

print(f"Resultados del LLM (Few-Shot Manual, m=5): {model_name}\n")
print(f"F1-Score macro: {round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2)}\n")
print(classification_report(dev_clickbait, predictions_clean))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Preparando 700 prompts...
Resultados del LLM (Few-Shot Manual, m=5): meta-llama/Llama-3.2-3B-Instruct

F1-Score macro: 50.14

              precision    recall  f1-score   support

   Clickbait       0.31      0.19      0.24       203
          No       0.71      0.82      0.77       497

    accuracy                           0.64       700
   macro avg       0.51      0.51      0.50       700
weighted avg       0.60      0.64      0.61       700



### Few-shot con ejemplos elegidos de manera automática

A continuación, incluiremos ejemplos seleccionados de manera automática, usando NearestNeighbor. Seguiremos los mismos criterios que en el parte anterior, en términos de selección de cantidad de ejemplos e inclusión de definición en la prompt.

modelo: Qwen/Qwen3-4B-Instruct-2507

Definición: No

m = 2 ejemplos elegidos mediante k-NN

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score, classification_report
import torch
import math
import gc
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')

model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")

embeddings_e5 = SentenceTransformer("intfloat/multilingual-e5-large")
train_headlines_emb = embeddings_e5.encode(train_headlines)
dev_headlines_emb = embeddings_e5.encode(dev_headlines)
print("Iniciando recuperación de vecinos con k-NN...")

nn_model = NearestNeighbors(n_neighbors=2, metric='cosine')
nn_model.fit(train_headlines_emb)

distances, neighbor_indices = nn_model.kneighbors(dev_headlines_emb)

print(f"Vecinos encontrados. {neighbor_indices.shape[0]} instancias de dev, {neighbor_indices.shape[1]} vecinos cada una.")

def crear_prompt_knn_fewshot(texto_nuevo, indices_vecinos):
    ejemplos = []
    for idx in indices_vecinos:
        ejemplos.append({
            "texto": train_headlines[idx],
            "label": train_clickbait[idx]
        })

    instruccion_prompt = 'Indicá si este titular es clickbait o no. Solamente respondé "Clickbait" o "No".\nAquí tienes algunos ejemplos:'

    messages = [
        {"role": "user", "content": instruccion_prompt}
    ]

    for ej in ejemplos:
        messages.append({"role": "user", "content": f'Titular: "{ej["texto"]}"'})
        messages.append({"role": "assistant", "content": ej["label"]})

    messages.append({"role": "user", "content": f'Titular: "{texto_nuevo}"'})

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text

print(f"Preparando {len(dev_headlines)} prompts dinámicos...")
all_formatted_prompts = []
for i in range(len(dev_headlines)):
    prompt = crear_prompt_knn_fewshot(dev_headlines[i], neighbor_indices[i])
    all_formatted_prompts.append(prompt)

print("Iniciando predicciones con LLM (Few-Shot k-NN)...")
batch_size = 16
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):
        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")

        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip().replace(".","")
            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"
            all_predictions_clean.append(clean_pred)

        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()

print("\nProceso completado. Evaluando resultados...")
predictions_clean = all_predictions_clean

print(f"\nResultados del LLM (Few-Shot k-NN, m=2): {model_name}\n")
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(dev_clickbait, predictions_clean))


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Iniciando recuperación de vecinos con k-NN...
Vecinos encontrados. 700 instancias de dev, 2 vecinos cada una.
Preparando 700 prompts dinámicos...
Iniciando predicciones con LLM (Few-Shot k-NN)...
Iniciando procesamiento en 44 lotes (batch_size=16)...
  -> Procesando lote 1 de 44...
  -> Procesando lote 2 de 44...
  -> Procesando lote 3 de 44...
  -> Procesando lote 4 de 44...
  -> Procesando lote 5 de 44...
  -> Procesando lote 6 de 44...
  -> Procesando lote 7 de 44...
  -> Procesando lote 8 de 44...
  -> Procesando lote 9 de 44...
  -> Procesando lote 10 de 44...
  -> Procesando lote 11 de 44...
  -> Procesando lote 12 de 44...
  -> Procesando lote 13 de 44...
  -> Procesando lote 14 de 44...
  -> Procesando lote 15 de 44...
  -> Procesando lote 16 de 44...
  -> Procesando lote 17 de 44...
  -> Procesando lote 18 de 44...
  -> Procesando lote 19 de 44...
  -> Procesando lote 20 de 44...
  -> Procesando lote 21 de 44...
  -> Procesando lote 22 de 44...
  -> Procesando lote 23 de 44...

modelo: Qwen/Qwen3-4B-Instruct-2507

Definicion: Si

m = 5 ejemplos elegidos mediante k-NN

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score, classification_report
import torch
import math
import gc
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')

model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")

embeddings_e5 = SentenceTransformer("intfloat/multilingual-e5-large")
train_headlines_emb = embeddings_e5.encode(train_headlines)
dev_headlines_emb = embeddings_e5.encode(dev_headlines)
print("Iniciando recuperación de vecinos con k-NN...")

nn_model = NearestNeighbors(n_neighbors=5, metric='cosine')
nn_model.fit(train_headlines_emb)

distances, neighbor_indices = nn_model.kneighbors(dev_headlines_emb)

print(f"Vecinos encontrados. {neighbor_indices.shape[0]} instancias de dev, {neighbor_indices.shape[1]} vecinos cada una.")

def crear_prompt_knn_fewshot(texto_nuevo, indices_vecinos):
    ejemplos = []
    for idx in indices_vecinos:
        ejemplos.append({
            "texto": train_headlines[idx],
            "label": train_clickbait[idx]
        })

    instruccion_prompt =  f"""
    Clickbait es una noticia que omite deliberadamente o tergiversa informacion con el objetivo de resultar mas atractiva y atraer clicks.
    Suelen hacer uso de puntuacion exagerada, uso de emojis y terminos con demasiada connotacion positiva o negativa, y tienen titulos sensacionales.”

    Indique si el siguiente titular es Clickbait o no:
    "{texto_nuevo}"

    Responde solamente "Clickbait" o "No".
    """
    messages = [
        {"role": "user", "content": instruccion_prompt}
    ]

    for ej in ejemplos:
        messages.append({"role": "user", "content": f'Titular: "{ej["texto"]}"'})
        messages.append({"role": "assistant", "content": ej["label"]})

    messages.append({"role": "user", "content": f'Titular: "{texto_nuevo}"'})

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text

print(f"Preparando {len(dev_headlines)} prompts dinámicos...")
all_formatted_prompts = []
for i in range(len(dev_headlines)):
    prompt = crear_prompt_knn_fewshot(dev_headlines[i], neighbor_indices[i])
    all_formatted_prompts.append(prompt)

print("Iniciando predicciones con LLM (Few-Shot k-NN)...")
batch_size = 16
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):
        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")

        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip().replace(".","")
            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"
            all_predictions_clean.append(clean_pred)

        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()

print("\nProceso completado. Evaluando resultados...")
predictions_clean = all_predictions_clean

print(f"\nResultados del LLM (Few-Shot k-NN, m=5): {model_name}\n")
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(dev_clickbait, predictions_clean))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Iniciando recuperación de vecinos con k-NN...
Vecinos encontrados. 700 instancias de dev, 5 vecinos cada una.
Preparando 700 prompts dinámicos...
Iniciando predicciones con LLM (Few-Shot k-NN)...
Iniciando procesamiento en 44 lotes (batch_size=16)...
  -> Procesando lote 1 de 44...
  -> Procesando lote 2 de 44...
  -> Procesando lote 3 de 44...
  -> Procesando lote 4 de 44...
  -> Procesando lote 5 de 44...
  -> Procesando lote 6 de 44...
  -> Procesando lote 7 de 44...
  -> Procesando lote 8 de 44...
  -> Procesando lote 9 de 44...
  -> Procesando lote 10 de 44...
  -> Procesando lote 11 de 44...
  -> Procesando lote 12 de 44...
  -> Procesando lote 13 de 44...
  -> Procesando lote 14 de 44...
  -> Procesando lote 15 de 44...
  -> Procesando lote 16 de 44...
  -> Procesando lote 17 de 44...
  -> Procesando lote 18 de 44...
  -> Procesando lote 19 de 44...
  -> Procesando lote 20 de 44...
  -> Procesando lote 21 de 44...
  -> Procesando lote 22 de 44...
  -> Procesando lote 23 de 44...

Al aumentar los ejemplos e incluir la definición, vemos que el rendimiento empmeoró, aunque no significativamente. Esto puede deberse a que los ejemplos seleccionados no se alinean con la definición incluida en la prompt. En particular, el recall de la clase "Clickbait" bajó significativamente, aunque la precisión de esa misma clase se mantuvo casi idéntica. Esto significa que, con más ejemplos y una definición, el modelo predijo menor cantidad de titulares como clickbait. En conjunto con los resultados obtenidos en la parte 3, esto apunta a que la definición de clickbait utilizada hace que los LLM se vuelvan más cautelosos al clasificar un titular como Clickbait.

---

modelo: ibm-granite/granite-4.0-h-350M

Definicion: no

ejemplos elegidos mediante NearestNeighbors m = 2

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score, classification_report
import torch
import math
import gc
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "ibm-granite/granite-4.0-h-350M"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")

embeddings_model = SentenceTransformer("intfloat/multilingual-e5-large")

train_headlines_emb = embeddings_model.encode(train_headlines)
dev_headlines_emb = embeddings_model.encode(dev_headlines)

m = 2

nn_model = NearestNeighbors(n_neighbors=m, metric='cosine')
nn_model.fit(train_headlines_emb)

distances, neighbor_indices = nn_model.kneighbors(dev_headlines_emb)

print(f"Vecinos encontrados. {neighbor_indices.shape[0]} instancias de dev, {neighbor_indices.shape[1]} vecinos cada una.")

def crear_prompt_knn_fewshot(texto_nuevo, indices_vecinos):
    ejemplos = []
    for idx in indices_vecinos:
        ejemplos.append({
            "texto": train_headlines[idx],
            "label": train_clickbait[idx]
        })

    instruccion_prompt = 'Indicá si este titular es clickbait o no. Solamente respondé "Clickbait" o "No".\nAquí tienes algunos ejemplos:'

    messages = [
        {"role": "user", "content": instruccion_prompt}
    ]

    for ej in ejemplos:
        messages.append({"role": "user", "content": f'Titular: "{ej["texto"]}"'})
        messages.append({"role": "assistant", "content": ej["label"]})

    messages.append({"role": "user", "content": f'Titular: "{texto_nuevo}"'})

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text

print(f"Preparando {len(dev_headlines)} prompts dinámicos...")
all_formatted_prompts = []
for i in range(len(dev_headlines)):
    prompt = crear_prompt_knn_fewshot(dev_headlines[i], neighbor_indices[i])
    all_formatted_prompts.append(prompt)

print("Iniciando predicciones con LLM (Few-Shot k-NN)...")
batch_size = 2
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):
        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")

        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip().replace(".","")
            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"
            all_predictions_clean.append(clean_pred)

        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()

print("\nProceso completado. Evaluando resultados...")
predictions_clean = all_predictions_clean

print(f"\nResultados del LLM (Few-Shot k-NN, m={m}): {model_name}\n")
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(dev_clickbait, predictions_clean))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Vecinos encontrados. 700 instancias de dev, 2 vecinos cada una.
Preparando 700 prompts dinámicos...
Iniciando predicciones con LLM (Few-Shot k-NN)...
Iniciando procesamiento en 350 lotes (batch_size=2)...
  -> Procesando lote 1 de 350...
  -> Procesando lote 2 de 350...
  -> Procesando lote 3 de 350...
  -> Procesando lote 4 de 350...
  -> Procesando lote 5 de 350...
  -> Procesando lote 6 de 350...
  -> Procesando lote 7 de 350...
  -> Procesando lote 8 de 350...
  -> Procesando lote 9 de 350...
  -> Procesando lote 10 de 350...
  -> Procesando lote 11 de 350...
  -> Procesando lote 12 de 350...
  -> Procesando lote 13 de 350...
  -> Procesando lote 14 de 350...
  -> Procesando lote 15 de 350...
  -> Procesando lote 16 de 350...
  -> Procesando lote 17 de 350...
  -> Procesando lote 18 de 350...
  -> Procesando lote 19 de 350...
  -> Procesando lote 20 de 350...
  -> Procesando lote 21 de 350...
  -> Procesando lote 22 de 350...
  -> Procesando lote 23 de 350...
  -> Procesando lote 2

modelo: ibm-granite/granite-4.0-h-350M

Definicion: Sí

ejemplos elegidos mediante NearestNeighbors m = 5

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score, classification_report
import torch
import math
import gc
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "ibm-granite/granite-4.0-h-350M"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")

embeddings_model = SentenceTransformer("intfloat/multilingual-e5-large")

train_headlines_emb = embeddings_model.encode(train_headlines)
dev_headlines_emb = embeddings_model.encode(dev_headlines)

m = 5

# usamos NearestNeighbors para encontrar los ejemplos mas cercanos (parecidos)
nn_model = NearestNeighbors(n_neighbors=m, metric='cosine')
nn_model.fit(train_headlines_emb)

# se encuentran los 'm' vecinos de train para cada instancia de dev
distances, neighbor_indices = nn_model.kneighbors(dev_headlines_emb)

print(f"Vecinos encontrados. {neighbor_indices.shape[0]} instancias de dev, {neighbor_indices.shape[1]} vecinos cada una.")

def crear_prompt_knn_fewshot(texto_nuevo, indices_vecinos):
    ejemplos = []
    for idx in indices_vecinos:
        ejemplos.append({
            "texto": train_headlines[idx],
            "label": train_clickbait[idx]
        })


    instruccion_prompt =  f"""
    Clickbait es una noticia que omite deliberadamente o tergiversa informacion con el objetivo de resultar mas atractiva y atraer clicks.
    Suelen hacer uso de puntuacion exagerada, uso de emojis y terminos con demasiada connotacion positiva o negativa, y tienen titulos sensacionales.”

    Indique si el siguiente titular es Clickbait o no:
    "{texto_nuevo}"

    Responde solamente "Clickbait" o "No".
    """

    messages = [
        {"role": "user", "content": instruccion_prompt}
    ]

    for ej in ejemplos:
        messages.append({"role": "user", "content": f'Titular: "{ej["texto"]}"'})
        messages.append({"role": "assistant", "content": ej["label"]})

    messages.append({"role": "user", "content": f'Titular: "{texto_nuevo}"'})

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text

print(f"Preparando {len(dev_headlines)} prompts dinámicos...")
all_formatted_prompts = []
for i in range(len(dev_headlines)):
    prompt = crear_prompt_knn_fewshot(dev_headlines[i], neighbor_indices[i])
    all_formatted_prompts.append(prompt)

print("Iniciando predicciones con LLM (Few-Shot k-NN)...")
batch_size = 2
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):
        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")

        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip().replace(".","")
            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"
            all_predictions_clean.append(clean_pred)

        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()

print("\nProceso completado. Evaluando resultados...")
predictions_clean = all_predictions_clean

print(f"\nResultados del LLM (Few-Shot k-NN, m={m}): {model_name}\n")
print(f"F1-Score macro: {str(round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(dev_clickbait, predictions_clean))

Vecinos encontrados. 700 instancias de dev, 5 vecinos cada una.
Preparando 700 prompts dinámicos...
Iniciando predicciones con LLM (Few-Shot k-NN)...
Iniciando procesamiento en 350 lotes (batch_size=2)...
  -> Procesando lote 1 de 350...
  -> Procesando lote 2 de 350...
  -> Procesando lote 3 de 350...
  -> Procesando lote 4 de 350...
  -> Procesando lote 5 de 350...
  -> Procesando lote 6 de 350...
  -> Procesando lote 7 de 350...
  -> Procesando lote 8 de 350...
  -> Procesando lote 9 de 350...
  -> Procesando lote 10 de 350...
  -> Procesando lote 11 de 350...
  -> Procesando lote 12 de 350...
  -> Procesando lote 13 de 350...
  -> Procesando lote 14 de 350...
  -> Procesando lote 15 de 350...
  -> Procesando lote 16 de 350...
  -> Procesando lote 17 de 350...
  -> Procesando lote 18 de 350...
  -> Procesando lote 19 de 350...
  -> Procesando lote 20 de 350...
  -> Procesando lote 21 de 350...
  -> Procesando lote 22 de 350...
  -> Procesando lote 23 de 350...
  -> Procesando lote 2

En el caso ibm-granite, nuevamente baja el recall de la clase Clickbait en comparación con el otro ejemplo que utiliza el mismo LLM, similar a lo que ha ocurrido en casos anteriores. Sin embargo, notamos una mejora tanto en la precisión de Clickbait como en el recall de la clase "No".

---

Modelo: Llama 3.2-3B

Definicion: No

Ejemplos elegidos: 2

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score, classification_report
import torch
import math
import gc
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

embeddings_model = SentenceTransformer("intfloat/multilingual-e5-large")

train_headlines_emb = embeddings_model.encode(train_headlines)
dev_headlines_emb = embeddings_model.encode(dev_headlines)

m = 2
nn_model = NearestNeighbors(n_neighbors=m, metric='cosine')
nn_model.fit(train_headlines_emb)

distances, neighbor_indices = nn_model.kneighbors(dev_headlines_emb)

def crear_prompt_knn_fewshot(texto_nuevo, indices_vecinos):
    ejemplos = []
    for idx in indices_vecinos:
        ejemplos.append({
            "texto": train_headlines[idx],
            "label": train_clickbait[idx]
        })

    instruccion = 'Indicá si este titular es clickbait o no. Solamente respondé "Clickbait" o "No".\nAquí tienes algunos ejemplos:'

    messages = [{"role": "user", "content": instruccion}]

    for ej in ejemplos:
        messages.append({"role": "user", "content": f'Titular: "{ej["texto"]}"'})
        messages.append({"role": "assistant", "content": ej["label"]})

    messages.append({"role": "user", "content": f'Titular: "{texto_nuevo}"'})

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

all_formatted_prompts = []
for i in range(len(dev_headlines)):
    prompt = crear_prompt_knn_fewshot(dev_headlines[i], neighbor_indices[i])
    all_formatted_prompts.append(prompt)

batch_size = 2
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):
        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean = pred.strip().split('\n')[0].strip().replace(".", "")
            if clean not in ["Clickbait", "No"]:
                if "clickbait" in clean.lower():
                    clean = "Clickbait"
                else:
                    clean = "No"
            all_predictions_clean.append(clean)

        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()

predictions_clean = all_predictions_clean

print(f"F1-Score macro: {round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2)}")
print(classification_report(dev_clickbait, predictions_clean))


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

F1-Score macro: 61.33
              precision    recall  f1-score   support

   Clickbait       0.54      0.33      0.41       203
          No       0.76      0.89      0.82       497

    accuracy                           0.72       700
   macro avg       0.65      0.61      0.61       700
weighted avg       0.70      0.72      0.70       700



Modelo: Llama 3.2-3B

Definición: Sí

Ejemplos elegidos: 5

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score, classification_report
import torch
import math
import gc
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

embeddings_model = SentenceTransformer("intfloat/multilingual-e5-large")

train_headlines_emb = embeddings_model.encode(train_headlines)
dev_headlines_emb = embeddings_model.encode(dev_headlines)

m = 5

nn_model = NearestNeighbors(n_neighbors=m, metric='cosine')
nn_model.fit(train_headlines_emb)

distances, neighbor_indices = nn_model.kneighbors(dev_headlines_emb)

def crear_prompt_knn_fewshot(texto_nuevo, indices_vecinos):
    ejemplos = []
    for idx in indices_vecinos:
        ejemplos.append({
            "texto": train_headlines[idx],
            "label": train_clickbait[idx]
        })

    instruccion_prompt = (
        'Indicá si este titular es clickbait o no. Se entiende por clickbait aquel '
        'titular sensacionalista o engañoso que tiene el objetivo de atraer clicks. '
        'Solamente respondé "Clickbait" o "No".\nAquí tienes algunos ejemplos:'
    )

    messages = [{"role": "user", "content": instruccion_prompt}]

    for ej in ejemplos:
        messages.append({"role": "user", "content": f'Titular: "{ej["texto"]}"'})
        messages.append({"role": "assistant", "content": ej["label"]})

    messages.append({"role": "user", "content": f'Titular: "{texto_nuevo}"'})

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

all_formatted_prompts = []
for i in range(len(dev_headlines)):
    prompt = crear_prompt_knn_fewshot(dev_headlines[i], neighbor_indices[i])
    all_formatted_prompts.append(prompt)

batch_size = 2
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):
        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip().replace(".", "")
            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                    clean_pred = "Clickbait"
                else:
                    clean_pred = "No"
            all_predictions_clean.append(clean_pred)

        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()

predictions_clean = all_predictions_clean

print(f"\nResultados del LLM (Few-Shot k-NN, m={m}): {model_name}\n")
print(f"F1-Score macro: {round(f1_score(dev_clickbait, predictions_clean, average='macro')*100, 2)}\n")
print(classification_report(dev_clickbait, predictions_clean))


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



Resultados del LLM (Few-Shot k-NN, m=5): meta-llama/Llama-3.2-3B-Instruct

F1-Score macro: 61.64

              precision    recall  f1-score   support

   Clickbait       0.58      0.31      0.40       203
          No       0.76      0.91      0.83       497

    accuracy                           0.73       700
   macro avg       0.67      0.61      0.62       700
weighted avg       0.71      0.73      0.71       700



En el caso de Llama 3.2-3B, el rendimiento en ambos modelos es casi idéntico. Sí notamos que el desmempeño al elegir ejemplos de forma automática es superior al de elegir ejemplos de forma aleatoria. Además, el desempeño también es mejor al obtenido a Llama3.2-3B en la parte 3.

##5️⃣Evaluación sobre el conjunto de *test*

En esta parte final del laboratorio, utilizaremos los modelos que obtuvieron los mejores resultados en el conjunto Dev para clasficar los titulares en el conjunto Train. De todas formas, incluimos modelos de las partes 3 y 4 a pesar de que no tuvieron un rendimiento competitivo. En particular, hicimos la siguiente selección de modelos (en línea con la nominación realizada en el informe):


*   LRBoW
*   SVCBoW
*   SVCe5
*   LRe5
*   Llama3.2Def
*   Granitek-NN




#Carga de datos Test

In [ ]:
%%capture
!wget https://raw.githubusercontent.com/pln-fing-udelar/pln-inco-resources/refs/heads/master/clickbait/iberlef2025/detection/TA1C_dataset_detection_test_gold.csv

with open("TA1C_dataset_detection_test_gold.csv","r") as f:
  test = [x for x in csv.reader(f)]
test_headlines = [x[4] for x in test[1:]]
test_clickbait = [x[5] for x in test[1:]]


LRBoW

In [ ]:
from sklearn.metrics import f1_score, classification_report
from sklearn.svm import SVC
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

bow_vectorizer = CountVectorizer()

LR = LogisticRegression(max_iter=1000, class_weight='balanced')

training_features = bow_vectorizer.fit_transform(train_headlines) # Se vectorizan los textos de train
LR.fit(training_features, train_clickbait) # Se entrena el clasificador usando los textos vectorizados

train_features = bow_vectorizer.transform(test_headlines) # Se vectorizan los textos de train
prediction = LR.predict(train_features) # Se predice si cada texto (ya vectorizado en la línea anterior) es o no clickbait

print(f"F1-Score macro: {str(round(f1_score(test_clickbait, prediction, average='macro')*100, 2))}\n")  # Se calcula el F1 y se pasa a escala de 100 para mayor legibilidad.
print(classification_report(test_clickbait, prediction)) # Se imprime un reporte de clasificación ya implementado en sklearn que, entre otras cosas, contiene al valor de F1 impreso en la línea anterior

F1-Score macro: 76.31

              precision    recall  f1-score   support

   Clickbait       0.66      0.62      0.64       167
          No       0.88      0.90      0.89       533

    accuracy                           0.83       700
   macro avg       0.77      0.76      0.76       700
weighted avg       0.83      0.83      0.83       700



SVCBoW

In [ ]:
svc = SVC(C=0.2, kernel='linear')

training_features = bow_vectorizer.fit_transform(train_headlines) # Se vectorizan los textos de train
svc.fit(training_features, train_clickbait) # Se entrena el clasificador usando los textos vectorizados

test_features = bow_vectorizer.transform(test_headlines) # Se vectorizan los textos de test
prediction = svc.predict(test_features) # Se predice si cada texto (ya vectorizado en la línea anterior) es o no clickbait

print(f"F1-Score macro: {str(round(f1_score(test_clickbait, prediction, average='macro')*100, 2))}\n")  # Se calcula el F1 y se pasa a escala de 100 para mayor legibilidad.
print(classification_report(test_clickbait, prediction)) # Se imprime un reporte de clasificación ya implementado en sklearn que, entre otras cosas, contiene al valor de F1 impreso en la línea anterior

F1-Score macro: 76.32

              precision    recall  f1-score   support

   Clickbait       0.70      0.57      0.63       167
          No       0.87      0.92      0.90       533

    accuracy                           0.84       700
   macro avg       0.78      0.75      0.76       700
weighted avg       0.83      0.84      0.83       700



SVCe5

In [ ]:
test_headlines_emb = embeddings_e5.encode(test_headlines)

svc = SVC(C=10, kernel='poly')

svc.fit(train_headlines_emb, train_clickbait) # Se ajusta el modelo para tomar como entrada los titulares de train y como salida los valores de clickbait de train.

predictions_svc = svc.predict(test_headlines_emb) # Una vez ajustado en los valores de train, se predicen nuevas instancias del conjunto de test.

print(f"F1-Score macro: {str(round(f1_score(test_clickbait, predictions_svc, average='macro')*100, 2))}\n") # Se calcula F1.
print(classification_report(test_clickbait, predictions_svc)) # Se hace el reporte de clasificación de sklearn.

F1-Score macro: 85.09

              precision    recall  f1-score   support

   Clickbait       0.75      0.81      0.78       167
          No       0.94      0.91      0.93       533

    accuracy                           0.89       700
   macro avg       0.84      0.86      0.85       700
weighted avg       0.89      0.89      0.89       700



LRe5

In [ ]:
LR = LogisticRegression(max_iter=1000, class_weight='balanced', C=20)

LR.fit(train_headlines_emb, train_clickbait) # Se ajusta el modelo para tomar como entrada los titulares de train y como salida los valores de clickbait de train.

predictions_LR = LR.predict(test_headlines_emb) # Una vez ajustado en los valores de train, se predicen nuevas instancias del conjunto de test.

print(f"F1-Score macro: {str(round(f1_score(test_clickbait, predictions_LR, average='macro')*100, 2))}\n") # Se calcula F1.
print(classification_report(test_clickbait, predictions_LR)) # Se hace el reporte de clasificación de sklearn.

F1-Score macro: 82.87

              precision    recall  f1-score   support

   Clickbait       0.67      0.86      0.75       167
          No       0.95      0.87      0.91       533

    accuracy                           0.86       700
   macro avg       0.81      0.86      0.83       700
weighted avg       0.88      0.86      0.87       700



*Llama3.2Def*

In [ ]:
import torch
import math
from sklearn.metrics import f1_score, classification_report
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")

print("Iniciando predicciones con LLM (Zero-Shot) sobre todo el conjunto test...")

def crear_prompt(texto):
    prompt = f"""
    Clickbait es una noticia que omite deliberadamente o tergiversa informacion con el objetivo de resultar mas atractiva y atraer clicks.
    Suelen hacer uso de puntuacion exagerada, uso de emojis y terminos con demasiada connotacion positiva o negativa, y tienen titulos sensacionales.”

    Indique si el siguiente titular es Clickbait o no:
    "{texto}"

    Responde solamente "Clickbait" o "No".
    """
    messages = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(f"Preparando {len(test_headlines)} prompts...")
all_formatted_prompts = [crear_prompt(texto) for texto in test_headlines]

batch_size = 16
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):
        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")
        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip()
            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"
            all_predictions_clean.append(clean_pred)

predictions_clean = all_predictions_clean

print("\nProceso completado. Evaluando resultados...\n")
print(f"Resultados del LLM: {model_name}\n")
print(f"F1-Score macro: {round(f1_score(test_clickbait, predictions_clean, average='macro')*100, 2)}\n")
print(classification_report(test_clickbait, predictions_clean))

print("\n--- Ejemplos de Predicciones ---")
for i in range(min(10, len(test_headlines))):
    print(f"Texto: {test_headlines[i]}")
    print(f"  -> Predicción: '{predictions_clean[i]}'")
    print(f"  -> Esperado:   '{test_clickbait[i]}'")


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Iniciando predicciones con LLM (Zero-Shot) sobre todo el conjunto test...
Preparando 700 prompts...
Iniciando procesamiento en 44 lotes (batch_size=16)...
  -> Procesando lote 1 de 44...
  -> Procesando lote 2 de 44...
  -> Procesando lote 3 de 44...
  -> Procesando lote 4 de 44...
  -> Procesando lote 5 de 44...
  -> Procesando lote 6 de 44...
  -> Procesando lote 7 de 44...
  -> Procesando lote 8 de 44...
  -> Procesando lote 9 de 44...
  -> Procesando lote 10 de 44...
  -> Procesando lote 11 de 44...
  -> Procesando lote 12 de 44...
  -> Procesando lote 13 de 44...
  -> Procesando lote 14 de 44...
  -> Procesando lote 15 de 44...
  -> Procesando lote 16 de 44...
  -> Procesando lote 17 de 44...
  -> Procesando lote 18 de 44...
  -> Procesando lote 19 de 44...
  -> Procesando lote 20 de 44...
  -> Procesando lote 21 de 44...
  -> Procesando lote 22 de 44...
  -> Procesando lote 23 de 44...
  -> Procesando lote 24 de 44...
  -> Procesando lote 25 de 44...
  -> Procesando lote 26 de 44

Granitek-NN

In [ ]:
from sklearn.neighbors import NearestNeighbors
import torch
import math
import gc
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "ibm-granite/granite-4.0-h-350M"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")

embeddings_model = SentenceTransformer("intfloat/multilingual-e5-large")

train_headlines_emb = embeddings_model.encode(train_headlines)
test_headlines_emb = embeddings_model.encode(test_headlines)

m = 2
nn_model = NearestNeighbors(n_neighbors=m, metric='cosine')
nn_model.fit(train_headlines_emb)

distances, neighbor_indices = nn_model.kneighbors(dev_headlines_emb)

print(f"Vecinos encontrados. {neighbor_indices.shape[0]} instancias de dev, {neighbor_indices.shape[1]} vecinos cada una.")

def crear_prompt_knn_fewshot(texto_nuevo, indices_vecinos):
    ejemplos = []
    for idx in indices_vecinos:
        ejemplos.append({
            "texto": train_headlines[idx],
            "label": train_clickbait[idx]
        })

    instruccion_prompt = 'Indicá si este titular es clickbait o no. Solamente respondé "Clickbait" o "No".\nAquí tienes algunos ejemplos:'

    messages = [
        {"role": "user", "content": instruccion_prompt}
    ]

    for ej in ejemplos:
        messages.append({"role": "user", "content": f'Titular: "{ej["texto"]}"'})
        messages.append({"role": "assistant", "content": ej["label"]})

    messages.append({"role": "user", "content": f'Titular: "{texto_nuevo}"'})

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_text

print(f"Preparando {len(test_headlines)} prompts dinámicos...")
all_formatted_prompts = []
for i in range(len(test_headlines)):
    prompt = crear_prompt_knn_fewshot(test_headlines[i], neighbor_indices[i])
    all_formatted_prompts.append(prompt)

print("Iniciando predicciones con LLM (Few-Shot k-NN)...")
batch_size = 2
all_predictions_clean = []
total_batches = math.ceil(len(all_formatted_prompts) / batch_size)

print(f"Iniciando procesamiento en {total_batches} lotes (batch_size={batch_size})...")

with torch.no_grad():
    for i in range(0, len(all_formatted_prompts), batch_size):
        print(f"  -> Procesando lote {i//batch_size + 1} de {total_batches}...")

        batch_prompts = all_formatted_prompts[i : i + batch_size]

        model_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.pad_token_id
        )

        output_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        predictions_raw = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        for pred in predictions_raw:
            clean_pred = pred.strip().split('\n')[0].strip().replace(".","")
            if clean_pred not in ["Clickbait", "No"]:
                if "clickbait" in clean_pred.lower():
                     clean_pred = "Clickbait"
                else:
                     clean_pred = "No"
            all_predictions_clean.append(clean_pred)

        del model_inputs, generated_ids, output_ids, predictions_raw, batch_prompts
        torch.cuda.empty_cache()
        gc.collect()

print("\nProceso completado. Evaluando resultados...")
predictions_clean = all_predictions_clean

print(f"\nResultados del LLM (Few-Shot k-NN, m={m}): {model_name}\n")
print(f"F1-Score macro: {str(round(f1_score(test_clickbait, predictions_clean, average='macro')*100, 2))}\\n")
print(classification_report(test_clickbait, predictions_clean))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/681M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Vecinos encontrados. 700 instancias de dev, 2 vecinos cada una.
Preparando 700 prompts dinámicos...
Iniciando predicciones con LLM (Few-Shot k-NN)...
Iniciando procesamiento en 350 lotes (batch_size=2)...
  -> Procesando lote 1 de 350...
  -> Procesando lote 2 de 350...
  -> Procesando lote 3 de 350...
  -> Procesando lote 4 de 350...
  -> Procesando lote 5 de 350...
  -> Procesando lote 6 de 350...
  -> Procesando lote 7 de 350...
  -> Procesando lote 8 de 350...
  -> Procesando lote 9 de 350...
  -> Procesando lote 10 de 350...
  -> Procesando lote 11 de 350...
  -> Procesando lote 12 de 350...
  -> Procesando lote 13 de 350...
  -> Procesando lote 14 de 350...
  -> Procesando lote 15 de 350...
  -> Procesando lote 16 de 350...
  -> Procesando lote 17 de 350...
  -> Procesando lote 18 de 350...
  -> Procesando lote 19 de 350...
  -> Procesando lote 20 de 350...
  -> Procesando lote 21 de 350...
  -> Procesando lote 22 de 350...
  -> Procesando lote 23 de 350...
  -> Procesando lote 2

En conclusión, como era esperable tras ver lo resultados obtenidos en el conjunto dev, los LLM tuvieron un desempeño notablemente inferior al de modelos estadísticos. Esto se debe en gran parte a que utilizamos LLMs con menor cantidad de parámetros que los LLM "estado del arte". En una posible continuación del obligatorio, sería interesante probar utilizar LLMs más grandes para saber qué tan cerca pueden estar (o si pueden superar) los modelos estadísticos. Además, creemos que la prompt utilizada condicionó fuertemente (en varios casos de forma negativa) el proceso de clasificación, y podría ser optimizada.